# Task 2: RAG vs RAG + Reranker

Сравнение скорости и качества контекста двух пайплайнов:
1. **Basic RAG** - FAISS + retriever (top-K)
2. **RAG + Reranker** - retriever (top-N) -> CrossEncoder reranker (top-K)

Видео: [ссылка](https://www.youtube.com/watch?v=lQTPc4qGSKk)

In [ ]:
import json
import os
import platform
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Для ноутбуков используем текущую директорию
NOTEBOOK_DIR = Path.cwd()
ARTIFACTS_DIR = NOTEBOOK_DIR / "artifacts"
CORPUS_PATH = NOTEBOOK_DIR.parent.parent / "data" / "video" / "video_2.txt"

# Конфигурация
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "google/gemini-2.0-flash-001")
EMBED_MODEL = "bge-m3"  # Через Ollama (ollama pull bge-m3)
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"

TOP_K = 5  # Финальное количество документов
TOP_N = 10  # Начальное количество для reranker
CHUNK_SIZE = 200
CHUNK_OVERLAP = 50

## Вспомогательные функции

In [ ]:
import dotenv

In [ ]:
dotenv.load_dotenv('../../.env')

In [ ]:
def get_device() -> str:
    """Автоопределение устройства для inference."""
    if platform.system() == "Darwin" and platform.machine() == "arm64":
        return "mps"  # Apple Silicon
    return "cpu"

def get_llm():
    """Возвращает LLM через OpenRouter."""
    return ChatOpenAI(
        model=OPENROUTER_MODEL,
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        temperature=0,
    )

print(f"LLM: {OPENROUTER_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"Reranker: {RERANKER_MODEL} on {get_device()}")

## Загрузка и разделение корпуса

In [ ]:
def load_corpus(path: Path) -> str:
    """Загружает корпус из файла."""
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def split_corpus(text: str, chunk_size: int = CHUNK_SIZE, chunk_overlap: int = CHUNK_OVERLAP) -> list:
    """Разбивает текст на чанки."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_text(text)

corpus_text = load_corpus(CORPUS_PATH)
chunks = split_corpus(corpus_text)

print(f"Corpus length: {len(corpus_text)} chars")
print(f"Chunks: {len(chunks)}")
print(f"Avg chunk length: {sum(len(c) for c in chunks) / len(chunks):.0f} chars")

## Построение FAISS индекса

In [ ]:
def build_faiss_index(chunks: list, embed_model: str = EMBED_MODEL) -> FAISS:
    """Создаёт FAISS индекс из чанков."""
    docs = [Document(page_content=chunk) for chunk in chunks]
    embeddings = OllamaEmbeddings(model=embed_model)
    return FAISS.from_documents(docs, embeddings)

t0 = time.perf_counter()
vector_store = build_faiss_index(chunks)
print(f"Index built in {time.perf_counter() - t0:.2f}s")

## Классы пайплайнов

In [ ]:
class BasicRAG:
    """Базовый RAG пайплайн без reranker."""

    def __init__(self, vector_store: FAISS, top_k: int = TOP_K):
        self.retriever = vector_store.as_retriever(search_kwargs={"k": top_k})
        self.top_k = top_k

    def retrieve(self, query: str) -> tuple:
        """Возвращает документы и время retrieval."""
        t0 = time.perf_counter()
        docs = self.retriever.invoke(query)
        t_retrieve = time.perf_counter() - t0
        return docs, {"retrieval": t_retrieve, "rerank": 0.0, "total": t_retrieve}

In [ ]:
class RAGWithReranker:
    """RAG пайплайн с CrossEncoder reranker."""

    def __init__(self, vector_store: FAISS, top_n: int = TOP_N, top_k: int = TOP_K, reranker_model: str = RERANKER_MODEL):
        self.retriever = vector_store.as_retriever(search_kwargs={"k": top_n})
        self.top_n = top_n
        self.top_k = top_k

        device = get_device()
        print(f"Loading reranker model: {reranker_model} on {device}")
        self.reranker = HuggingFaceCrossEncoder(model_name=reranker_model, model_kwargs={"device": device})

    def retrieve(self, query: str) -> tuple:
        """Возвращает документы и времена retrieval + rerank."""
        t0 = time.perf_counter()
        docs = self.retriever.invoke(query)
        t_retrieve = time.perf_counter() - t0

        t1 = time.perf_counter()
        scores_info = []
        if len(docs) > self.top_k:
            pairs = [[query, doc.page_content] for doc in docs]
            scores = self.reranker.score(pairs)
            scores_info = list(zip(range(len(docs)), scores))
            ranked_indices = np.argsort(scores)[::-1][: self.top_k]
            docs = [docs[i] for i in ranked_indices]

        t_rerank = time.perf_counter() - t1
        t_total = time.perf_counter() - t0

        return docs, {"retrieval": t_retrieve, "rerank": t_rerank, "total": t_total, "scores_info": scores_info}

## Генерация вопросов из корпуса

In [ ]:
def generate_questions_from_chunks(chunks: list, n_questions: int = 15, llm=None) -> list:
    """Генерирует вопросы из чанков корпуса с помощью LLM."""
    import random
    if llm is None:
        llm = get_llm()

    selected_chunks = random.sample(chunks, min(n_questions, len(chunks)))
    questions = []
    print(f"Generating {len(selected_chunks)} questions from corpus...")

    for i, chunk in enumerate(selected_chunks):
        prompt = f"""На основе следующего текста, составь 1 вопрос, на который можно ответить, используя этот текст.

Текст: {chunk[:500]}

Вопрос (только сам вопрос, без номера):"""

        try:
            response = llm.invoke(prompt)
            question = response.content.strip().strip('"')
            questions.append(question)
            print(f"  [{i+1}/{len(selected_chunks)}] {question[:60]}...")
        except Exception as e:
            print(f"  Error generating question {i+1}: {e}")
            continue

    return questions

# Генерация вопросов
llm = get_llm()
test_questions = generate_questions_from_chunks(chunks, n_questions=15, llm=llm)
print(f"\nGenerated {len(test_questions)} questions")

## Инициализация пайплайнов

In [ ]:
basic_rag = BasicRAG(vector_store, top_k=TOP_K)
rag_rerank = RAGWithReranker(vector_store, top_n=TOP_N, top_k=TOP_K)
print("Pipelines initialized")

## Функции генерации ответов

In [ ]:
def generate_answer(query: str, context: str, llm=None) -> tuple:
    """Генерирует ответ через OpenRouter."""
    if llm is None:
        llm = get_llm()

    prompt = f"""Контекст:
{context}

Вопрос: {query}

Ответь на вопрос, используя только информацию из контекста."""

    t0 = time.perf_counter()
    try:
        response = llm.invoke(prompt)
        answer = response.content if hasattr(response, "content") else str(response)
    except Exception as e:
        answer = f"Error: {e}"
    t_gen = time.perf_counter() - t0

    return answer, t_gen

def run_pipeline(pipeline, query: str, llm=None) -> dict:
    """Запускает пайплайн и собирает метрики."""
    if llm is None:
        llm = get_llm()

    docs, times = pipeline.retrieve(query)
    context = "\n\n".join(doc.page_content for doc in docs)
    answer, t_gen = generate_answer(query, context, llm)

    return {
        "query": query,
        "answer": answer,
        "context_length": len(context),
        "num_docs": len(docs),
        "times": {**{k: v for k, v in times.items() if k != "scores_info"}, "generation": t_gen},
        "total_time": times["total"] + t_gen,
    }

## Сравнение пайплайнов

In [ ]:
def compare_pipelines(questions: list, basic_rag: BasicRAG, rag_rerank: RAGWithReranker) -> dict:
    """Сравнивает два пайплайна на списке вопросов."""
    results_basic = []
    results_rerank = []

    llm = get_llm()

    for i, question in enumerate(questions):
        print(f"\n[{i + 1}/{len(questions)}] {question[:50]}...")

        print("  Running Basic RAG...")
        result_basic = run_pipeline(basic_rag, question, llm)
        results_basic.append(result_basic)
        print(f"    Time: {result_basic['total_time']:.3f}s")

        print("  Running RAG + Reranker...")
        result_rerank = run_pipeline(rag_rerank, question, llm)
        results_rerank.append(result_rerank)
        print(f"    Time: {result_rerank['total_time']:.3f}s")

    def aggregate(results: list) -> dict:
        times = [r["times"] for r in results]
        return {
            "avg_retrieval_time": sum(t["retrieval"] for t in times) / len(times),
            "avg_rerank_time": sum(t["rerank"] for t in times) / len(times),
            "avg_generation_time": sum(t["generation"] for t in times) / len(times),
            "avg_total_time": sum(r["total_time"] for r in results) / len(results),
            "avg_context_length": sum(r["context_length"] for r in results) / len(results),
        }

    return {
        "basic_rag": {"aggregate": aggregate(results_basic), "details": results_basic},
        "rag_reranker": {"aggregate": aggregate(results_rerank), "details": results_rerank},
    }

# Запуск сравнения
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
comparison = compare_pipelines(test_questions, basic_rag, rag_rerank)

## График сравнения

In [ ]:
def plot_comparison(comparison: dict, save_path: Path = None):
    """Строит графики сравнения."""
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    basic = comparison["basic_rag"]["aggregate"]
    rerank = comparison["rag_reranker"]["aggregate"]

    labels = ["Basic RAG", "RAG + Reranker"]
    colors = ["steelblue", "coral"]

    # Retrieval Time
    ax = axes[0]
    values = [basic["avg_retrieval_time"], rerank["avg_retrieval_time"]]
    ax.bar(labels, values, color=colors)
    ax.set_title("Avg Retrieval Time")
    ax.set_ylabel("Seconds")
    for i, v in enumerate(values):
        ax.annotate(f"{v:.3f}s", (i, v), ha="center", va="bottom")

    # Retrieval+Rerank Time
    ax = axes[1]
    values = [basic["avg_total_time"], rerank["avg_total_time"]]
    ax.bar(labels, values, color=colors)
    ax.set_title("Avg Retrieval+Rerank Time")
    ax.set_ylabel("Seconds")
    for i, v in enumerate(values):
        ax.annotate(f"{v:.3f}s", (i, v), ha="center", va="bottom")

    # Generation Time
    ax = axes[2]
    values = [basic["avg_generation_time"], rerank["avg_generation_time"]]
    ax.bar(labels, values, color=colors)
    ax.set_title("Avg Generation Time")
    ax.set_ylabel("Seconds")
    for i, v in enumerate(values):
        ax.annotate(f"{v:.3f}s", (i, v), ha="center", va="bottom")

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=100)
        print(f"Saved: {save_path}")
    plt.show()

plot_comparison(comparison, ARTIFACTS_DIR / "rag_comparison.png")

## График разбивки по времени

In [ ]:
def plot_time_breakdown(comparison: dict, save_path: Path = None):
    """Строит stacked bar chart с разбивкой по времени."""
    fig, ax = plt.subplots(figsize=(10, 6))

    basic = comparison["basic_rag"]["aggregate"]
    rerank = comparison["rag_reranker"]["aggregate"]

    labels = ["Basic RAG", "RAG + Reranker"]
    x = np.arange(len(labels))
    width = 0.5

    retrieval = [basic["avg_retrieval_time"], rerank["avg_retrieval_time"]]
    rerank_time = [0, rerank["avg_rerank_time"]]
    generation = [basic["avg_generation_time"], rerank["avg_generation_time"]]

    bars1 = ax.bar(x, retrieval, width, label="Retrieval", color="steelblue")
    bars2 = ax.bar(x, rerank_time, width, bottom=retrieval, label="Rerank", color="coral")
    bars3 = ax.bar(x, generation, width, bottom=[r + rr for r, rr in zip(retrieval, rerank_time)], label="Generation", color="green")

    ax.set_ylabel("Time (seconds)")
    ax.set_title("Time Breakdown by Pipeline Stage")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()

    for i, (r, rr, g) in enumerate(zip(retrieval, rerank_time, generation)):
        total = r + rr + g
        ax.annotate(f"Total: {total:.2f}s", (i, total), ha="center", va="bottom", fontsize=10)

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=100)
        print(f"Saved: {save_path}")
    plt.show()

plot_time_breakdown(comparison, ARTIFACTS_DIR / "time_breakdown.png")

## Результаты

In [ ]:
import pandas as pd

basic = comparison["basic_rag"]["aggregate"]
rerank = comparison["rag_reranker"]["aggregate"]

df = pd.DataFrame([
    {"Pipeline": "Basic RAG", **basic},
    {"Pipeline": "RAG + Reranker", **rerank},
])

print("=== Results ===")
df

## Сохранение результатов

In [ ]:
output = {
    "config": {
        "corpus_path": str(CORPUS_PATH),
        "num_chunks": len(chunks),
        "embed_model": EMBED_MODEL,
        "reranker_model": RERANKER_MODEL,
        "device": get_device(),
        "llm_model": OPENROUTER_MODEL,
        "top_k": TOP_K,
        "top_n": TOP_N,
    },
    "comparison": comparison,
}

with open(ARTIFACTS_DIR / "rag_comparison.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2, default=str)

print(f"Saved: {ARTIFACTS_DIR / 'rag_comparison.json'}")
print("\n=== Task 2 Complete ===")